# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_2_DEMO_Session_4_external_evaluation_pipelines_langsmith.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





<!-- NOTEBOOK_METADATA source: "Jupyter Notebook" title: "Evaluate LangSmith Agent Traces with an External Evaluation Pipeline" description: "This notebook shows how to push representative traces to LangSmith (US API), fetch them for scoring, optionally attach feedback, and turn critical failures into a replayable benchmark set for model comparison." category: "Evaluation" sidebarTitle: "External Evaluation Pipelines (LangSmith)"-->

# Evaluate LangSmith Agent Traces with an External Evaluation Pipeline

This cookbook shows how to use **LangSmith** as the trace store, filtering layer, and optional feedback layer for an external evaluation pipeline. We target the **US-hosted** LangSmith API (`https://api.smith.langchain.com`).

Tracing tells you what happened: latency, tool calls, retries, and outputs. It does not tell you whether a changed model still meets your application-specific quality bar. That is why we push traces, fetch representative runs, score them outside the core observability UI if we want to, optionally attach **feedback** to runs, and promote critical failures into a replayable benchmark set before model replacement.


The strongest practical design choice is to avoid making the notebook depend on LLM judging alone.

Instead, mix three kinds of signals:

- Hard checks for things you can verify directly, such as retry budget, schema validity, tool success, required handoff labels, or numeric tolerances.
- Structured rubric checks for dimensions like sufficiency or completeness.
- Trace-level judge checks when the path matters, not just the final answer.

That combination matches how LangSmith is typically most useful in production: store runs in a project, filter representative examples, and visualize or query feedback. The scoring logic itself should model good evaluation practice rather than focus on stylistic traits like tone or joyfulness.

<iframe
  width="100%"
  className="aspect-[3230/2160] rounded mt-10"
  src="https://www.youtube-nocookie.com/embed/rHfME8KDmIw?si=V4m8smxZ219AKmOU"
  title="YouTube video player"
  frameborder="0"
  allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share"
  referrerpolicy="strict-origin-when-cross-origin"
  allowFullScreen
></iframe>


---

By the end of this cookbook, you will be able to:

- Push synthetic support-agent traces to LangSmith with tags and structured outputs.
- Fetch representative **root runs** from a LangSmith project using time-window and tag filters (via the SDK).
- Define multi-layer evaluation criteria for agent workflows and outcomes.
- Score traces externally with hard checks, rubric checks, or both (including `deepeval` metrics).
- Optionally attach **feedback** to runs in LangSmith for later filtering in the UI.
- Promote critical failures into a replayable benchmark set.
- Compare candidate model versions on that benchmark set before redeployment.
- See how **RULER-style group ranking** complements per-trace scores for open-ended tasks.
- Read how **conditional tracing** (`tracing_context`) ties observability to app logic (compliance, tenants, cost).

Conceptually, we will implement the following architecture:

```mermaid
sequenceDiagram
participant Application
participant LangSmith
participant EvalPipeline as External Eval Pipeline
participant BenchmarkSet
participant Experiment
actor User

Application ->> LangSmith: Ingest root runs with tags and outputs
LangSmith ->> EvalPipeline: list_runs with tag and time filters
EvalPipeline ->> EvalPipeline: Run hard checks and judge-based scoring
EvalPipeline ->> BenchmarkSet: Persist critical failures and benchmark cases
EvalPipeline ->> LangSmith: Optionally create_feedback on runs
Experiment ->> BenchmarkSet: Load regression cases
Experiment ->> EvalPipeline: Score candidate model runs
LangSmith ->> User: Optionally explore runs and feedback in the UI
```

---


**Note**: While we’re using a Jupyter notebook for this cookbook, in production you'd use your preferred orchestration tool. Just make sure to extract the code into a .py file and ensure all dependencies are available at runtime.



## (Prep-work) Load representative support-agent traces into LangSmith

In this demo, we will avoid a generic text-style example and instead use a lightweight support-agent workflow. Each synthetic trace will include enough structure to evaluate agent behavior: retry counts, handoff targets, required steps, tool outcomes, and the final answer.

In production, you would ingest real traces first (for example from your instrumented app). Here we create representative synthetic traces so the rest of the notebook can focus on the evaluation pipeline itself.


You can get your LangSmith API key from [LangSmith Settings](https://smith.langchain.com/settings) and your OpenAI API key from [OpenAI](https://platform.openai.com/api-keys).

Use the **US** API host unless your workspace is in the EU region: set `LANGCHAIN_ENDPOINT=https://api.smith.langchain.com`. For EU, use `https://eu.api.smith.langchain.com` instead.

The **LangSmith project name** for this notebook is set in the credentials code cell (variable `LANGSMITH_PROJECT_NAME`), not via a `LANGCHAIN_PROJECT` environment variable.

If you see **`403 Forbidden` on `/sessions`** when logging runs: use project **`default`** (the notebook default), **create the project in the [LangSmith UI](https://smith.langchain.com)** before using a custom name, confirm your API key belongs to the same organization, and set **`LANGCHAIN_ENDPOINT`** to the **EU** host (`https://eu.api.smith.langchain.com`) if your workspace is in the EU region.


## Documentation index and conditional tracing

The LangChain and LangSmith documentation is indexed at [docs.langchain.com/llms.txt](https://docs.langchain.com/llms.txt). Use that file to discover pages before deep-linking.

### Conditional tracing and app logic

When tracing is enabled globally (for example via `LANGSMITH_TRACING` / `LANGCHAIN_TRACING_V2`), spans and runs are sent to LangSmith by default. In production you usually need **request-level** control without rewriting the whole stack: skip tracing for PII-heavy paths, route tenants to different projects, or turn tracing off for low-value traffic to manage cost.

In Python, the [`tracing_context`](https://reference.langchain.com/python/langsmith/run_helpers/tracing_context) context manager overrides global settings for code executed **inside** the `with` block. Typical patterns:

- **Disable tracing** for sensitive payloads: `with ls.tracing_context(enabled=False): ...`
- **Route per tenant or region**: `with ls.tracing_context(project_name="client-acme", tags=[...], metadata={...}): ...`
- **Feature flags**: enable tracing only when your app logic says the request is worth observability spend

Precedence (highest first): `tracing_context` → programmatic configuration → environment variables. See the official guide: [Conditional tracing](https://docs.langchain.com/langsmith/conditional-tracing).

**Sampling vs conditional tracing**: conditional tracing is **deterministic** (you decide per request). [Sampling](https://docs.langchain.com/langsmith/sample-traces) is **probabilistic** for high-volume cost control. You can combine both.

Related: [Trace without environment variables](https://docs.langchain.com/langsmith/trace-without-env-vars), [Mask inputs and outputs](https://docs.langchain.com/langsmith/mask-inputs-outputs), [Add metadata and tags to traces](https://docs.langchain.com/langsmith/add-metadata-tags).

Below, this notebook uses a **fixed** project name in code plus `@traceable` for the demo harness. In a real app, you would often wrap handler entrypoints in `tracing_context` so tracing follows **your** routing and compliance rules.

Example (not executed later in this notebook — adapt in your app):

```python
import langsmith as ls
from langsmith import traceable

@traceable
def handle(customer_id: str, text: str) -> str:
    return text

# Skip tracing entirely (PII, zero-retention tenant, etc.)
with ls.tracing_context(enabled=False):
    handle("regulated-tenant", "...")

# Route to a tenant- or region-specific project with tags and metadata
with ls.tracing_context(
    enabled=True,
    project_name="acme-support",
    tags=["production", "region-us"],
    metadata={"customer_id": "acme"},
):
    handle("acme", "...")
```



In [ ]:
%pip install langsmith openai deepeval


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.3 MB/s eta 0:00:00


In [ ]:
# Store your secrets in a local .env file or your notebook environment.
# Do not hardcode credentials directly in the notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# US-hosted LangSmith (default). For EU, set LANGCHAIN_ENDPOINT=https://eu.api.smith.langchain.com
LS_API_URL = os.getenv("LANGCHAIN_ENDPOINT", "https://api.smith.langchain.com")
os.environ["LANGCHAIN_ENDPOINT"] = LS_API_URL

api_key = os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY")
if not api_key:
    raise ValueError(
        "Set LANGCHAIN_API_KEY (or LANGSMITH_API_KEY) in your environment before running this notebook."
    )
os.environ["LANGCHAIN_API_KEY"] = api_key
os.environ["LANGSMITH_API_KEY"] = api_key

# LangSmith project for this cookbook — set here (no LANGCHAIN_PROJECT env var required).
# Use "default" unless you created a dedicated project in the UI first. A non-existent name often
# causes LangSmithError 403 on GET /sessions when the SDK resolves the project.
# EU workspace: set LANGCHAIN_ENDPOINT=https://eu.api.smith.langchain.com above.
LANGSMITH_PROJECT_NAME = "default"
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT_NAME

# Enable tracing for @traceable (LangSmith accepts LANGCHAIN_* and LANGSMITH_* mirrors)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_TRACING_V2"] = "true"


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not found.")
    OPENAI_API_KEY = input("Enter OPENAI_API_KEY: ").strip()
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY is set.")
print("LangSmith credentials are set.")
print(f"LangSmith project (this notebook): {os.environ['LANGCHAIN_PROJECT']}")


OPENAI_API_KEY is set.
LangSmith credentials are set.
LangSmith project (this notebook): default


Let's define a small set of representative support scenarios. Each case includes the workflow we expect, the required operational step, the correct handoff target, and a reference answer we can later use for correctness checks.

In [ ]:
cases = [
    {
        "case_id": "refund_double_charge",
        "workflow": "billing",
        "agent_type": "triage_agent",
        "customer_request": "I was charged twice for order 8129. Please fix it.",
        "required_step": "lookup_order",
        "expected_handoff": "billing_specialist",
        "reference_answer": "I found the duplicate charge on order 8129 and handed this to billing for refund review.",
        "max_retries": 2,
        "simulated_retry_count": 1,
        "simulated_handoff_target": "billing_specialist",
        "simulated_steps_completed": ["lookup_order", "verify_duplicate_charge", "handoff_to_billing"],
        "simulated_tool_calls": [
            {"name": "lookup_order", "status": "success"},
            {"name": "create_billing_handoff", "status": "success"}
        ],
        "simulated_final_answer": "I found the duplicate charge on order 8129 and handed your case to billing for refund review. You should receive an update within one business day."
    },
    {
        "case_id": "shipping_status_no_handoff_needed",
        "workflow": "shipping",
        "agent_type": "triage_agent",
        "customer_request": "Where is my order 9912 and when should it arrive?",
        "required_step": "lookup_shipment",
        "expected_handoff": "none",
        "reference_answer": "Order 9912 is in transit and should arrive tomorrow based on the shipment lookup.",
        "max_retries": 1,
        "simulated_retry_count": 0,
        "simulated_handoff_target": "none",
        "simulated_steps_completed": ["lookup_shipment", "respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_shipment", "status": "success"}
        ],
        "simulated_final_answer": "I checked shipment 9912 and it is currently in transit. The latest carrier estimate shows delivery tomorrow."
    },
    {
        "case_id": "reset_password_wrong_handoff",
        "workflow": "account_access",
        "agent_type": "triage_agent",
        "customer_request": "I cannot log in after changing my password. Can someone help me reset access?",
        "required_step": "lookup_account_status",
        "expected_handoff": "account_security",
        "reference_answer": "I checked your account status and handed the case to account security so they can restore access safely.",
        "max_retries": 1,
        "simulated_retry_count": 0,
        "simulated_handoff_target": "billing_specialist",
        "simulated_steps_completed": ["lookup_account_status", "handoff_to_billing"],
        "simulated_tool_calls": [
            {"name": "lookup_account_status", "status": "success"},
            {"name": "create_billing_handoff", "status": "success"}
        ],
        "simulated_final_answer": "I reviewed the account issue and escalated it for follow-up. Please wait for the next update."
    },
    {
        "case_id": "cancel_subscription_missing_step",
        "workflow": "billing",
        "agent_type": "triage_agent",
        "customer_request": "Please cancel my subscription at the end of this billing cycle.",
        "required_step": "lookup_subscription",
        "expected_handoff": "none",
        "reference_answer": "I found your subscription and queued it to cancel at the end of the current billing cycle.",
        "max_retries": 1,
        "simulated_retry_count": 0,
        "simulated_handoff_target": "none",
        "simulated_steps_completed": ["respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_subscription", "status": "failed"}
        ],
        "simulated_final_answer": "Your subscription is all set and should stop soon."
    },
    {
        "case_id": "invoice_request_retry_exceeded",
        "workflow": "billing",
        "agent_type": "billing_agent",
        "customer_request": "Can you send me the invoice for March?",
        "required_step": "lookup_invoice",
        "expected_handoff": "none",
        "reference_answer": "I found your March invoice and sent it to the email on file.",
        "max_retries": 1,
        "simulated_retry_count": 3,
        "simulated_handoff_target": "none",
        "simulated_steps_completed": ["lookup_invoice", "email_invoice"],
        "simulated_tool_calls": [
            {"name": "lookup_invoice", "status": "success"},
            {"name": "email_invoice", "status": "success"}
        ],
        "simulated_final_answer": "I sent your March invoice to the email on file."
    },
    {
        "case_id": "wrong_fact_in_final_answer",
        "workflow": "returns",
        "agent_type": "returns_agent",
        "customer_request": "What is the return deadline for order 4481?",
        "required_step": "lookup_return_policy",
        "expected_handoff": "none",
        "reference_answer": "Order 4481 is eligible for return within 30 days of delivery.",
        "max_retries": 1,
        "simulated_retry_count": 0,
        "simulated_handoff_target": "none",
        "simulated_steps_completed": ["lookup_return_policy", "respond_to_customer"],
        "simulated_tool_calls": [
            {"name": "lookup_return_policy", "status": "success"}
        ],
        "simulated_final_answer": "Order 4481 can be returned within 14 days of delivery."
    }
]

for case in cases:
    print(f"{case['case_id']}: workflow={case['workflow']}, agent_type={case['agent_type']}")


refund_double_charge: workflow=billing, agent_type=triage_agent
shipping_status_no_handoff_needed: workflow=shipping, agent_type=triage_agent
reset_password_wrong_handoff: workflow=account_access, agent_type=triage_agent
cancel_subscription_missing_step: workflow=billing, agent_type=triage_agent
invoice_request_retry_exceeded: workflow=billing, agent_type=billing_agent
wrong_fact_in_final_answer: workflow=returns, agent_type=returns_agent


Next, we will log one synthetic **root run** per case to LangSmith using `@traceable`. The point is not to build a realistic support agent in this prep section. The point is to create representative runs with the kinds of workflow signals an external evaluator can inspect later: retries, tool outcomes, handoffs, and the final answer.

Tags (`ext_eval_pipelines`, `support_agent`, workflow, agent type) let us retrieve the right slice later with `Client.list_runs`.


In [ ]:
import json
import os

from langsmith import Client, traceable

ls_client = Client(api_url=os.environ["LANGCHAIN_ENDPOINT"])
project_name = os.environ["LANGCHAIN_PROJECT"]


def log_support_trace(case):
    payload = {
        "case_id": case["case_id"],
        "workflow": case["workflow"],
        "agent_type": case["agent_type"],
        "customer_request": case["customer_request"],
        "required_step": case["required_step"],
        "expected_handoff": case["expected_handoff"],
        "reference_answer": case["reference_answer"],
        "max_retries": case["max_retries"],
        "retry_count": case["simulated_retry_count"],
        "handoff_target": case["simulated_handoff_target"],
        "steps_completed": case["simulated_steps_completed"],
        "tool_calls": case["simulated_tool_calls"],
        "final_answer": case["simulated_final_answer"],
    }

    @traceable(
        name=f"Support case: {case['case_id']}",
        tags=[
            "ext_eval_pipelines",
            "support_agent",
            case["workflow"],
            case["agent_type"],
        ],
        project_name=project_name,
    )
    def _emit_payload():
        return payload

    return _emit_payload()


for case in cases:
    logged_trace = log_support_trace(case)
    print(
        json.dumps(
            {
                "case_id": logged_trace["case_id"],
                "retry_count": logged_trace["retry_count"],
                "handoff_target": logged_trace["handoff_target"],
                "final_answer": logged_trace["final_answer"],
            },
            indent=2,
        )
    )

ls_client.flush()
print("Traces flushed to LangSmith.")


{
  "case_id": "refund_double_charge",
  "retry_count": 1,
  "handoff_target": "billing_specialist",
  "final_answer": "I found the duplicate charge on order 8129 and handed your case to billing for refund review. You should receive an update within one business day."
}
{
  "case_id": "shipping_status_no_handoff_needed",
  "retry_count": 0,
  "handoff_target": "none",
  "final_answer": "I checked shipment 9912 and it is currently in transit. The latest carrier estimate shows delivery tomorrow."
}
{
  "case_id": "reset_password_wrong_handoff",
  "retry_count": 0,
  "handoff_target": "billing_specialist",
  "final_answer": "I reviewed the account issue and escalated it for follow-up. Please wait for the next update."
}
{
  "case_id": "cancel_subscription_missing_step",
  "retry_count": 0,
  "handoff_target": "none",
  "final_answer": "Your subscription is all set and should stop soon."
}
{
  "case_id": "invoice_request_retry_exceeded",
  "retry_count": 3,
  "handoff_target": "none",
  "f

Now you should see representative support-agent runs in your LangSmith project. Open [smith.langchain.com](https://smith.langchain.com), select the project matching **`LANGSMITH_PROJECT_NAME`** in the credentials cell (the default is **`default`**), and browse **Runs** — filter by tag `ext_eval_pipelines` if helpful.

This notebook does not embed static screenshots; the LangSmith UI evolves frequently, so follow the live project view instead.


## 1. Why tracing alone is not enough

Tracing shows you operational behavior such as latency, retries, tool calls, and outputs. That is necessary, but it is still incomplete: when you swap models or adjust orchestration, you also need to know whether the system still satisfies workflow-specific quality expectations like taking the right handoff, completing the required step, and giving a sufficient and correct final answer.

## 2. Fetch representative production traces

Fetching runs from LangSmith is straightforward with `Client.list_runs`. The important part is not just pulling any recent runs, but selecting representative ones. In practice, you will usually filter by a time window and then narrow the set by tags, workflow, agent type, or another slice that matters for the release decision you are about to make.

The code below first asks LangSmith for **root** runs with `has(tags, "ext_eval_pipelines")`. Some workspaces or SDK versions return an empty list even when traces exist, so it **falls back** to listing recent roots and matching that tag in Python. If the project is very busy (for example the **`default`** project), it also widens the API time window to the last seven days.


In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone

from langsmith import Client

BATCH_SIZE = 10
TOTAL_TRACES = 100
EVAL_TAG = "ext_eval_pipelines"
WORKFLOW_FILTERS = {"billing", "shipping", "returns", "account_access"}
AGENT_TYPE_FILTERS = {"triage_agent", "billing_agent", "returns_agent"}

LS_API_URL = os.environ["LANGCHAIN_ENDPOINT"]
PROJECT_NAME = os.environ["LANGCHAIN_PROJECT"]
ls_client = Client(api_url=LS_API_URL)

now = datetime.now(timezone.utc)
five_am_today = datetime(now.year, now.month, now.day, 5, 0, tzinfo=timezone.utc)
five_am_yesterday = five_am_today - timedelta(days=1)
seven_days_ago = now - timedelta(days=7)


def parse_trace_payload(run):
    out = run.outputs
    if out is None:
        raise TypeError("Run has no outputs")
    if isinstance(out, dict):
        if set(out.keys()) == {"output"} and isinstance(out["output"], dict):
            return out["output"]
        if isinstance(out.get("output"), dict) and "case_id" in out["output"]:
            return out["output"]
        if "case_id" in out and "workflow" in out:
            return out
    if isinstance(out, str):
        return json.loads(out)
    raise TypeError("Unsupported run output format")


def is_support_case_trace(run):
    trace_name = getattr(run, "name", "") or ""
    if trace_name.startswith("Support case:"):
        return True

    try:
        payload = parse_trace_payload(run)
    except Exception:
        return False

    required_keys = {"workflow", "agent_type", "required_step", "final_answer"}
    return required_keys.issubset(payload.keys())


def matches_eval_filters(run):
    try:
        payload = parse_trace_payload(run)
    except Exception:
        return False

    return (
        payload.get("workflow") in WORKFLOW_FILTERS
        and payload.get("agent_type") in AGENT_TYPE_FILTERS
    )


def run_tags(run):
    tags = getattr(run, "tags", None) or []
    if tags:
        return list(tags)
    extra = getattr(run, "extra", None) or {}
    return list(extra.get("tags") or [])


def run_start_utc(run):
    st = run.start_time
    if st is None:
        return None
    if isinstance(st, str):
        st = datetime.fromisoformat(st.replace("Z", "+00:00"))
    if st.tzinfo is None:
        st = st.replace(tzinfo=timezone.utc)
    return st


def in_time_window(run, from_ts, to_ts):
    st = run_start_utc(run)
    if st is None:
        return True
    return from_ts <= st <= to_ts


def fetch_support_traces(from_timestamp, to_timestamp, verbose=True):
    """Pull root runs; prefer server-side tag filter, fall back if it returns nothing."""

    def query_runs(*, tag_filter: bool, start_time, limit: int):
        kw = dict(
            project_name=PROJECT_NAME,
            start_time=start_time,
            is_root=True,
            limit=limit,
        )
        if tag_filter:
            kw["filter"] = f'has(tags, "{EVAL_TAG}")'
        return list(ls_client.list_runs(**kw))

    # 1) Tag filter + API start_time (fast path when it matches your LangSmith version)
    candidates = query_runs(tag_filter=True, start_time=from_timestamp, limit=TOTAL_TRACES)
    if verbose and not candidates:
        print(
            "Note: list_runs with has(tags, ...) returned 0 rows; "
            "retrying without the server tag filter and matching tags in Python."
        )

    if not candidates:
        candidates = query_runs(tag_filter=False, start_time=from_timestamp, limit=TOTAL_TRACES)

    # 2) Still empty — widen API window to 7d (busy "default" projects may need this)
    if not candidates:
        if verbose:
            print("Note: widening API start_time to the last 7 days to find recent roots.")
        candidates = query_runs(
            tag_filter=False, start_time=seven_days_ago, limit=TOTAL_TRACES
        )

    if not candidates:
        if verbose:
            print(
                "Note: listing the most recent root runs with no start_time filter "
                f"(cap {TOTAL_TRACES}); check project name if this is still empty."
            )
        candidates = list(
            ls_client.list_runs(
                project_name=PROJECT_NAME,
                is_root=True,
                limit=TOTAL_TRACES,
            )
        )

    windowed = [r for r in candidates if in_time_window(r, from_timestamp, to_timestamp)]
    if not windowed and candidates:
        tag_only = [r for r in candidates if EVAL_TAG in run_tags(r)]
        if tag_only:
            if verbose:
                print(
                    "Note: no runs fell in the requested time window; "
                    "using tag-matched runs from the query batch instead."
                )
            windowed = tag_only
    tagged = [r for r in windowed if EVAL_TAG in run_tags(r)]
    raw_batch = tagged[:BATCH_SIZE] if tagged else windowed[:BATCH_SIZE]

    support_traces = [t for t in raw_batch if is_support_case_trace(t)]
    filtered_traces = [t for t in support_traces if matches_eval_filters(t)]
    return raw_batch, support_traces, filtered_traces


raw_batch, support_traces, traces_batch = fetch_support_traces(
    from_timestamp=five_am_yesterday,
    to_timestamp=now,
)

if not traces_batch:
    raw_batch, support_traces, traces_batch = fetch_support_traces(
        from_timestamp=seven_days_ago,
        to_timestamp=now,
    )
    print("No matching traces found in the last day, so the search was widened to the last 7 days.")

print(f"Tagged traces fetched: {len(raw_batch)}")
print(f"Support-case traces found: {len(support_traces)}")
print(f"Representative traces in first batch: {len(traces_batch)}")

if not traces_batch:
    raise ValueError(
        "No support-agent traces matched the evaluation filters. "
        "Re-run the prep cell in this kernel, then this cell. "
        f"Confirm runs land in project {PROJECT_NAME!r} (same LANGCHAIN_PROJECT as tracing) "
        f"and runs include tag {EVAL_TAG!r} (see LangSmith UI)."
    )





Tagged traces fetched: 10
Support-case traces found: 10
Representative traces in first batch: 10


## 3. Define multi-layer evaluation criteria

For this notebook, we will score each trace across five dimensions:

- `retry_budget_respected`
- `correct_handoff`
- `required_step_completed`
- `final_answer_sufficient`
- `final_answer_correct`

The first three are ideal hard checks. The last two are where structured rubric checks or trace-level judges are often useful.

Hard checks should do as much of the work as possible:

- `retry_count <= max_retries`
- required tool calls succeeded
- required handoff label is present when escalation is needed
- required workflow step appears in the trajectory

Then add rubric-based checks for sufficiency and correctness where a strict programmatic comparison is not enough.

Trace-level judging fits when the path matters, not just the final answer. This is where a trajectory judge or a system like RULER is a strong conceptual fit: compare trajectories against an explicit goal-achievement rubric when failure is only visible in the full path.

In the code below, we keep the production pattern simple: hard checks for workflow compliance, plus external judge-style scoring for sufficiency and correctness.

In [ ]:
from deepeval.metrics import (
    GEval,
    TaskCompletionMetric,
    ArgumentCorrectnessMetric,
    StepEfficiencyMetric,
)
from deepeval.test_case import LLMTestCaseParams, LLMTestCase, ToolCall

JUDGE_MODEL = "gpt-4o"
DEFAULT_THRESHOLD = 0.7


def normalize_retry_count(value, fallback):
    try:
        return int(value)
    except (TypeError, ValueError):
        return fallback


def normalize_steps_completed(value):
    if isinstance(value, list):
        return [str(item) for item in value]
    if isinstance(value, str):
        return [value]
    return []


def normalize_list(value):
    if isinstance(value, list):
        return value
    if value is None:
        return []
    return [value]


def normalize_tool_calls(value):
    normalized_calls = []
    for call in normalize_list(value):
        if isinstance(call, dict):
            normalized_calls.append(
                {
                    "name": str(call.get("name", "unknown_tool")),
                    "status": str(call.get("status", "unknown")),
                    "input": call.get("input", {}),
                    "output": call.get("output", {}),
                    "description": str(call.get("description", "")),
                }
            )
        else:
            normalized_calls.append(
                {
                    "name": str(call),
                    "status": "unknown",
                    "input": {},
                    "output": {},
                    "description": "",
                }
            )
    return normalized_calls


def normalize_payload(payload):
    normalized = dict(payload)
    normalized["retry_count"] = normalize_retry_count(
        normalized.get("retry_count"), normalized.get("max_retries", 0) + 1
    )
    normalized["steps_completed"] = normalize_steps_completed(normalized.get("steps_completed"))
    normalized["tool_calls"] = normalize_tool_calls(normalized.get("tool_calls"))
    normalized["handoff_target"] = str(normalized.get("handoff_target", "none"))
    normalized["expected_handoff"] = str(normalized.get("expected_handoff", "none"))
    normalized["required_step"] = str(normalized.get("required_step", ""))
    normalized["final_answer"] = str(normalized.get("final_answer", ""))
    normalized["reference_answer"] = str(normalized.get("reference_answer", ""))
    normalized["customer_request"] = str(normalized.get("customer_request", ""))
    normalized["workflow"] = str(normalized.get("workflow", "unknown"))
    normalized["agent_type"] = str(normalized.get("agent_type", "unknown"))
    normalized["max_retries"] = normalize_retry_count(normalized.get("max_retries"), 0)
    normalized["task_definition"] = str(
        normalized.get(
            "task_definition",
            f"Resolve {normalized.get('workflow','support')} request with required step {normalized.get('required_step','unknown')}."
        )
    )
    return normalized


def payload_to_tool_calls(payload_tools):
    return [
        ToolCall(
            name=tool.get("name", "unknown_tool"),
            description=tool.get("description", ""),
            input=tool.get("input", {}),
            output=tool.get("output", {}),
        )
        for tool in payload_tools
    ]


def build_llm_test_case(payload):
    payload = normalize_payload(payload)
    return LLMTestCase(
        input=payload["customer_request"],
        actual_output=payload["final_answer"],
        tools_called=payload_to_tool_calls(payload["tool_calls"]),
    )


def retry_budget_respected(payload):
    payload = normalize_payload(payload)
    return payload["retry_count"] <= payload["max_retries"]


def correct_handoff(payload):
    payload = normalize_payload(payload)
    return payload["handoff_target"] == payload["expected_handoff"]


def required_step_completed(payload):
    payload = normalize_payload(payload)
    return payload["required_step"] in payload["steps_completed"]


def tool_success(payload):
    payload = normalize_payload(payload)
    relevant_calls = [call for call in payload["tool_calls"] if payload["required_step"] in call["name"]]
    if not relevant_calls:
        return False
    return all(call["status"] == "success" for call in relevant_calls)


def trajectory_summary(payload):
    payload = normalize_payload(payload)
    tool_lines = [f"- {call['name']}: {call['status']}" for call in payload["tool_calls"]]
    return f"""
Customer request: {payload['customer_request']}
Workflow: {payload['workflow']}
Agent type: {payload['agent_type']}
Retry count: {payload['retry_count']} of {payload['max_retries']}
Expected handoff: {payload['expected_handoff']}
Observed handoff: {payload['handoff_target']}
Required step: {payload['required_step']}
Completed steps: {', '.join(payload['steps_completed'])}
Tool calls:
{chr(10).join(tool_lines) if tool_lines else '- none recorded'}
Final answer: {payload['final_answer']}
Reference answer: {payload['reference_answer']}
""".strip()


def final_answer_sufficient(payload):
    payload = normalize_payload(payload)
    metric = GEval(
        name="final_answer_sufficient",
        criteria=(
            "Determine whether the final answer is sufficient for the customer's request. "
            "The answer should reflect workflow context and include a clear next action when needed."
        ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=JUDGE_MODEL,
    )
    test_case = LLMTestCase(input=trajectory_summary(payload), actual_output=payload["final_answer"])
    metric.measure(test_case)
    return {"score": metric.score, "reason": metric.reason}


def final_answer_correct(payload):
    payload = normalize_payload(payload)
    metric = GEval(
        name="final_answer_correct",
        criteria="Assess whether the final answer is correct given the workflow details and reference answer.",
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ],
        model=JUDGE_MODEL,
    )
    test_case = LLMTestCase(
        input=trajectory_summary(payload),
        actual_output=payload["final_answer"],
        expected_output=payload["reference_answer"],
    )
    metric.measure(test_case)
    return {"score": metric.score, "reason": metric.reason}


def score_task_completion(payload):
    p = normalize_payload(payload)
    metric = TaskCompletionMetric(
        threshold=DEFAULT_THRESHOLD,
        model=JUDGE_MODEL,
        task=p["task_definition"],
        include_reason=True,
        async_mode=False,
    )
    metric.measure(build_llm_test_case(p))
    return {"score": float(metric.score), "reason": metric.reason}


def score_argument_correctness(payload):
    metric = ArgumentCorrectnessMetric(
        threshold=DEFAULT_THRESHOLD,
        model=JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    )
    metric.measure(build_llm_test_case(payload))
    return {"score": float(metric.score), "reason": metric.reason}


def score_step_efficiency(payload):
    payload = normalize_payload(payload)
    metric = StepEfficiencyMetric(
        threshold=DEFAULT_THRESHOLD,
        model=JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    )
    try:
        metric.measure(build_llm_test_case(payload))
        return {"score": float(metric.score), "reason": metric.reason}
    except UnboundLocalError as exc:
        # DeepEval StepEfficiencyMetric can fail internally for some payload shapes.
        # Deterministic fallback: penalize retries and extra steps beyond a minimal plan.
        required = payload.get("required_step", "")
        steps = payload.get("steps_completed", [])
        max_retries = max(int(payload.get("max_retries", 0)), 0)
        retries = max(int(payload.get("retry_count", 0)), 0)

        base_plan_len = 2  # required step + response
        observed_len = len(steps)
        missing_required_penalty = 0.5 if required and required not in steps else 0.0
        extra_step_penalty = max(0, observed_len - base_plan_len) * 0.1
        retry_penalty = 0 if max_retries == 0 else min(retries / max_retries, 1.0) * 0.4

        score = max(0.0, min(1.0, 1.0 - missing_required_penalty - extra_step_penalty - retry_penalty))
        reason = (
            "Fallback score used because StepEfficiencyMetric failed internally "
            f"({exc.__class__.__name__}). "
            f"Derived from required-step presence, extra steps ({observed_len}), and retry usage ({retries}/{max_retries})."
        )
        return {"score": float(score), "reason": reason}



Wrapping evaluation logic in functions keeps the pipeline easy to test and version. Each function takes a trace payload and returns a boolean, categorical, or numeric score that can later be pushed back to LangSmith as **feedback** if you choose.


In [ ]:
example_payload = parse_trace_payload(traces_batch[0])

example_scores = {
    "retry_budget_respected": retry_budget_respected(example_payload),
    "correct_handoff": correct_handoff(example_payload),
    "required_step_completed": required_step_completed(example_payload),
    "required_tool_succeeded": tool_success(example_payload),
    "final_answer_sufficient": final_answer_sufficient(example_payload),
    "final_answer_correct": final_answer_correct(example_payload),
}

example_scores

Output()

Output()

{'retry_budget_respected': True,
 'correct_handoff': True,
 'required_step_completed': True,
 'required_tool_succeeded': True,
 'final_answer_sufficient': {'score': 0.44777806741033865,
  'reason': "The Actual Output addresses the customer's request by providing a return deadline for order 4481, aligning with step 1. However, it does not match the Reference answer, indicating a discrepancy in the return policy information. The response is concise and directly answers the request, fulfilling step 4, but lacks additional workflow context or a suggested next action, which affects steps 2 and 3."},
 'final_answer_correct': {'score': 0.39385741192404006,
  'reason': "The Actual Output does not match the Expected Output, as it states a 14-day return period instead of the correct 30-day period. The Input aligns with the workflow details, as the required step 'lookup_return_policy' was completed successfully. However, the discrepancy in the return period is not justified by the Input or workfl

## 4. Score traces externally

Use external judge logic, hard checks, or both. The important thing is that the score logic lives outside the tracing system, so it can evolve with your application. LangSmith can store the run and optionally store **feedback** on that run, but your evaluator defines what good behavior means.

You can use any evaluation library. `deepeval` is sufficient here for independent numeric checks, while RULER-style trajectory judging is a good next step when you need explicit **group-relative** comparisons (especially for open-ended answers).


In [ ]:
evaluated_traces = []

for trace in traces_batch:
    payload = normalize_payload(parse_trace_payload(trace))

    sufficiency = final_answer_sufficient(payload)
    correctness = final_answer_correct(payload)
    task_completion = score_task_completion(payload)
    argument_correctness = score_argument_correctness(payload)
    step_efficiency = score_step_efficiency(payload)

    evaluated_traces.append(
        {
            "trace_id": str(trace.id),
            "case_id": payload["case_id"],
            "workflow": payload["workflow"],
            "agent_type": payload["agent_type"],
            "payload": payload,
            "scores": {
                "retry_budget_respected": float(retry_budget_respected(payload)),
                "correct_handoff": float(correct_handoff(payload)),
                "required_step_completed": float(required_step_completed(payload)),
                "required_tool_succeeded": float(tool_success(payload)),
                "final_answer_sufficient": sufficiency["score"],
                "final_answer_correct": correctness["score"],
                "task_completion": task_completion["score"],
                "argument_correctness": argument_correctness["score"],
                "step_efficiency": step_efficiency["score"],
            },
            "reasons": {
                "final_answer_sufficient": sufficiency["reason"],
                "final_answer_correct": correctness["reason"],
                "task_completion": task_completion["reason"],
                "argument_correctness": argument_correctness["reason"],
                "step_efficiency": step_efficiency["reason"],
            },
        }
    )

evaluated_traces[0]


Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

{'trace_id': '019d2066-0bf9-7d82-90ca-a6dc210fcb27',
 'case_id': 'wrong_fact_in_final_answer',
 'workflow': 'returns',
 'agent_type': 'returns_agent',
 'payload': {'agent_type': 'returns_agent',
  'case_id': 'wrong_fact_in_final_answer',
  'customer_request': 'What is the return deadline for order 4481?',
  'expected_handoff': 'none',
  'final_answer': 'Order 4481 can be returned within 14 days of delivery.',
  'handoff_target': 'none',
  'max_retries': 1,
  'reference_answer': 'Order 4481 is eligible for return within 30 days of delivery.',
  'required_step': 'lookup_return_policy',
  'retry_count': 0,
  'steps_completed': ['lookup_return_policy', 'respond_to_customer'],
  'tool_calls': [{'name': 'lookup_return_policy',
    'status': 'success',
    'input': {},
    'output': {},
    'description': ''}],
  'workflow': 'returns',
  'task_definition': 'Resolve returns request with required step lookup_return_policy.'},
 'scores': {'retry_budget_respected': 1.0,
  'correct_handoff': 1.0,


For LLM-based checks, keep the reasons. They are useful for debugging and for reviewing borderline cases later. For hard checks, the boolean itself is often enough, but for judge-based scores the explanation becomes part of the audit trail.

## 5. Optional: attach feedback to runs in LangSmith


This step is optional. If you want the pipeline to remain fully independent, you can stop after exporting runs, scoring them externally, and writing your benchmark set to disk or another store.

LangSmith becomes especially useful again if you want score visualization and filtering in the same system that stores the traces. In that case, attach booleans, numeric scores, and comments back to the original **run** using the SDK (`Client.create_feedback`) for follow-up analysis in the UI.


In [ ]:
WRITE_SCORES_TO_LANGSMITH = False

if WRITE_SCORES_TO_LANGSMITH:
    for evaluated in evaluated_traces:
        rid = evaluated["trace_id"]
        for metric_name, metric_value in evaluated["scores"].items():
            kwargs = {
                "run_id": rid,
                "key": metric_name,
                "score": float(metric_value),
            }
            if metric_name in evaluated["reasons"]:
                kwargs["comment"] = evaluated["reasons"][metric_name]
            ls_client.create_feedback(**kwargs)

    print(f"Pushed feedback for {len(evaluated_traces)} runs.")
else:
    print(
        "Skipping LangSmith feedback write-back. Set WRITE_SCORES_TO_LANGSMITH = True to enable it."
    )


Skipping LangSmith feedback write-back. Set WRITE_SCORES_TO_LANGSMITH = True to enable it.


## 6. Promote critical failures into a replayable benchmark set

This is the missing bridge from tracing to regression testing. A trace becomes much more valuable once a critical failure is promoted into a replayable benchmark case with its workflow metadata, reference answer, prior scores, and evaluator notes.

That benchmark set is also where you can start tracking richer text-quality criteria. Independent metrics like `deepeval` can score sufficiency and correctness one trace at a time. Later, if you want to compare several candidate trajectories for the same benchmark case, you can add a relative judge such as RULER on top.

In [ ]:
import json

critical_failures = [
    evaluated
    for evaluated in evaluated_traces
    if (
        evaluated["scores"]["retry_budget_respected"] < 1
        or evaluated["scores"]["correct_handoff"] < 1
        or evaluated["scores"]["required_step_completed"] < 1
        or evaluated["scores"]["required_tool_succeeded"] < 1
        or evaluated["scores"]["final_answer_sufficient"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["final_answer_correct"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["task_completion"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["argument_correctness"] < DEFAULT_THRESHOLD
        or evaluated["scores"]["step_efficiency"] < DEFAULT_THRESHOLD
    )
]

benchmark_set_path = "benchmark_set_langsmith.jsonl"

with open(benchmark_set_path, "w", encoding="utf-8") as benchmark_file:
    for failure in critical_failures:
        benchmark_file.write(
            json.dumps(
                {
                    "case_id": failure["case_id"],
                    "workflow": failure["workflow"],
                    "agent_type": failure["agent_type"],
                    "benchmark_type": "critical_failure_regression",
                    "payload": failure["payload"],
                    "reference_answer": failure["payload"]["reference_answer"],
                    "scores": failure["scores"],
                    "reasons": failure["reasons"],
                    "text_quality_dimensions": [
                        "final_answer_sufficient",
                        "final_answer_correct",
                        "task_completion",
                        "argument_correctness",
                        "step_efficiency",
                    ],
                }
            )
            + "\n"
        )

print(f"Benchmark cases written: {len(critical_failures)}")
print(f"Benchmark set path: {benchmark_set_path}")


Benchmark cases written: 10
Benchmark set path: benchmark_set_langsmith.jsonl


## 7. Compare candidate model versions on the benchmark set

Now the benchmark set can be used as a lightweight regression suite. Instead of promoting a new model because it looks good on a few spot checks, run the benchmark cases again, rescore them externally, and compare aggregate results before redeployment.

Review means and drill into per-case failures in this notebook or by re-loading the benchmark JSONL in your own tooling.


The code cell below replays the benchmark cases against candidate models and summarizes the resulting scores. In practice, you would run this step before changing the production model, the routing logic, or the tool orchestration.

This still scores each run independently. That is useful, but sometimes not enough. If several candidate trajectories all look plausible and you want a judge to rank them relative to one another on goal achievement or text quality, add a relative evaluator such as RULER on top of the benchmark set.

In [ ]:
import statistics

from openai import OpenAI

openai_client = OpenAI()

with open(benchmark_set_path, "r", encoding="utf-8") as benchmark_file:
    benchmark_cases = [json.loads(line) for line in benchmark_file]


def run_candidate_agent(payload, model_name):
    payload = normalize_payload(payload)

    prompt = f"""
You are a customer support agent. Return JSON only with these keys:
retry_count, handoff_target, steps_completed, tool_calls, final_answer.

Constraints:
- steps_completed must be a JSON array of strings.
- tool_calls must be a JSON array of objects, each with keys name, status, input, output, description.
- retry_count must be an integer.
- handoff_target must be a string.
- final_answer must be a string.

Customer request: {payload['customer_request']}
Workflow: {payload['workflow']}
Required step: {payload['required_step']}
Expected handoff: {payload['expected_handoff']}
Reference answer: {payload['reference_answer']}
""".strip()

    response = openai_client.chat.completions.create(
        model=model_name,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
        timeout=120,
    )

    candidate = json.loads(response.choices[0].message.content)
    return normalize_payload(
        {
            **payload,
            "retry_count": candidate.get("retry_count", payload["max_retries"] + 1),
            "handoff_target": candidate.get("handoff_target", "none"),
            "steps_completed": candidate.get("steps_completed", []),
            "tool_calls": candidate.get("tool_calls", []),
            "final_answer": candidate.get("final_answer", ""),
        }
    )


def evaluate_candidate_model(model_name, benchmark_cases):
    model_scores = []
    for benchmark_case in benchmark_cases:
        payload = run_candidate_agent(benchmark_case["payload"], model_name)
        sufficiency = final_answer_sufficient(payload)
        correctness = final_answer_correct(payload)
        task_completion = score_task_completion(payload)
        argument_correctness = score_argument_correctness(payload)
        step_efficiency = score_step_efficiency(payload)
        model_scores.append(
            {
                "retry_budget_respected": float(retry_budget_respected(payload)),
                "correct_handoff": float(correct_handoff(payload)),
                "required_step_completed": float(required_step_completed(payload)),
                "required_tool_succeeded": float(tool_success(payload)),
                "final_answer_sufficient": sufficiency["score"],
                "final_answer_correct": correctness["score"],
                "task_completion": task_completion["score"],
                "argument_correctness": argument_correctness["score"],
                "step_efficiency": step_efficiency["score"],
            }
        )

    if not model_scores:
        return {"model": model_name, "cases": 0}

    return {
        "model": model_name,
        "cases": len(model_scores),
        **{k: statistics.mean(s[k] for s in model_scores) for k in model_scores[0].keys()},
    }


model_comparison = [
    evaluate_candidate_model("gpt-5.4", benchmark_cases),
    evaluate_candidate_model("gpt-5.4-mini", benchmark_cases),
]

model_comparison


Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

[{'model': 'gpt-5.4',
  'cases': 10,
  'retry_budget_respected': 1.0,
  'correct_handoff': 1.0,
  'required_step_completed': 0.4,
  'required_tool_succeeded': 0.4,
  'final_answer_sufficient': 0.999470363254377,
  'final_answer_correct': 0.9999999999999999,
  'task_completion': 0.77,
  'argument_correctness': 0.0,
  'step_efficiency': 0.61},
 {'model': 'gpt-5.4-mini',
  'cases': 10,
  'retry_budget_respected': 1.0,
  'correct_handoff': 1.0,
  'required_step_completed': 0.9,
  'required_tool_succeeded': 0.0,
  'final_answer_sufficient': 0.9923726642647523,
  'final_answer_correct': 0.9893146250363849,
  'task_completion': 0.75,
  'argument_correctness': 0.2,
  'step_efficiency': 0.94}]

## 8. RULER moved to separate notebook

Comparative answer ranking with RULER now lives in `ch09_ruler_trace_answer_ranking_langsmith.ipynb`.
This notebook keeps the single-trace evaluation loop only.


In [ ]:
print("Use ch09_ruler_trace_answer_ranking_langsmith.ipynb for comparative RULER ranking.")


Use ch09_ruler_trace_answer_ranking_langsmith.ipynb for comparative RULER ranking.


This gives you the full production loop for agent quality: instrument the system, ingest runs into LangSmith, fetch representative traces, score them on workflow and outcome dimensions, optionally attach feedback to those runs, convert important failures into a benchmark set, and compare candidate model versions against that set before rollout.

When independent metrics are not enough, the same benchmark set can also support **group-relative** ranking with a trajectory judge such as RULER. In a real deployment, schedule this pipeline in CI or your orchestration platform, and run the benchmark comparison whenever you change the model, prompts, tools, or routing logic.
